# SUNO v2 — train the classifier head (Part A)

This notebook does **Part A only**: it trains the small classifier that sits on top of the frozen YAMNet model, writes `suno_audio_classifier_v2.tflite` + `labels_v2.json`, and replays the app's alert rule on held-out clips. It never retrains YAMNet.

**Run every cell top to bottom.** The guards stop the notebook early with a plain-English message instead of letting a bad file train a meaningless model.

Before you start:
1. `MyDrive/SUNO_dataset/clips/` has the 5 `a*` folders and 18 `n*` folders, spelled exactly as in the guide.
2. `MyDrive/SUNO_dataset/yamnet.tflite` is the **~16 MB** file from the app (`assets/ml/yamnet.tflite`), not the old empty placeholder.
3. Runtime → Change runtime type → **CPU** is fine. No GPU needed.

## Step 0 — Mount Drive and point at the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/SUNO_dataset'
print('ROOT =', ROOT)

## Step 0b — Hard checks before anything is read as training data

If this cell stops, fix the message before continuing. Do not skip it.

In [ ]:
import os, re, glob, shutil

assert os.path.isdir(ROOT), f'Dataset folder not found: {ROOT}'

yamnet_path = os.path.join(ROOT, 'yamnet.tflite')
assert os.path.isfile(yamnet_path), (
    'SUNO_dataset/yamnet.tflite is missing. Upload the app copy of '
    'assets/ml/yamnet.tflite (about 16 MB).'
)
yamnet_bytes = os.path.getsize(yamnet_path)
assert yamnet_bytes > 15_000_000, (
    f'SUNO_dataset/yamnet.tflite is only {yamnet_bytes:,} bytes. '
    'That is the old placeholder. Replace it with the ~16 MB file from '
    'assets/ml/yamnet.tflite before running anything else.'
)

clips_dir = os.path.join(ROOT, 'clips')
assert os.path.isdir(clips_dir), f'Missing clips folder: {clips_dir}'
folders = sorted(
    d for d in os.listdir(clips_dir)
    if os.path.isdir(os.path.join(clips_dir, d))
)

EXPECTED_ALERT = {
    'a01_scream_loud',
    'a02_gun_shots',
    'a03_glass_break',
    'a04_crash_heavy',
    'a05_alarm_home',
}
missing = EXPECTED_ALERT - set(folders)
assert not missing, (
    f'Missing alert folders: {sorted(missing)}. Folder names become class '
    'names, so fix the spellings in Drive before training.'
)
bad_names = [d for d in folders if not re.match(r'^[an]\d{2}_', d)]
assert not bad_names, (
    f'Unexpected folder names: {bad_names}. Every folder must start with '
    'a01_... through a05_... or n01_... through n18_....'
)
no_effect = [d for d in folders if d.startswith('n')]
assert len(no_effect) >= 18, (
    f'Only {len(no_effect)} no-effect folders found; expected 18.'
)

print(f'yamnet.tflite OK: {yamnet_bytes:,} bytes')
print(f'{len(folders)} class folders:', ', '.join(folders))

## Step 2 — Install dependencies

In [ ]:
!pip -q install soundfile librosa scikit-learn pandas

## Step 3 — Config

`ALLOW_LOW_SOURCES = False` stops the run when a class has fewer than 40 distinct recordings. If a folder really is that small, the model will memorise those few recordings instead of learning the sound.

In [ ]:
import os, re, json, glob, hashlib, random, time, datetime
import numpy as np, pandas as pd, soundfile as sf, librosa
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

SR, WIN, HOP, EMB = 16000, 15360, 7680, 1024   # must match the app exactly
SEED = 42
ALLOW_LOW_SOURCES = False
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

ALERT_WEIGHTS = {
    'a01_scream_loud': 50,
    'a02_gun_shots':   50,
    'a03_glass_break': 35,
    'a04_crash_heavy': 40,
    'a05_alarm_home':  35,
}
MIN_HITS = {
    'a01_scream_loud': 2, 'a02_gun_shots': 1, 'a03_glass_break': 1,
    'a04_crash_heavy': 1, 'a05_alarm_home': 2,
}
MARGIN = 0.25

print('config ready')

## Step 4 — Build the manifest

Converts every readable clip to 16 kHz WAV, cuts long no-effect recordings into 8-second pieces, and drops exact duplicates across the whole dataset.

In [ ]:
AUDIO_EXT = {'.wav', '.mp3', '.m4a', '.aac', '.ogg', '.oga', '.opus', '.flac',
             '.3gp', '.amr', '.wma', '.mp4', '.webm'}

def group_of(stem):
    m = re.match(r'^\d+-(\d+)-[A-Za-z]-\d+$', stem)
    if m: return f'esc50-{m.group(1)}'
    m = re.match(r'^(\d+)-\d+-\d+-\d+$', stem)
    if m: return f'us8k-{m.group(1)}'
    return stem.split('__')[0]

shutil.rmtree(f'{ROOT}/_wav', ignore_errors=True)
os.makedirs(f'{ROOT}/_wav', exist_ok=True)

rows, skipped, seen = [], [], {}
for cls_dir in sorted(glob.glob(f'{ROOT}/clips/*/')):
    cls = os.path.basename(os.path.normpath(cls_dir))
    os.makedirs(f'{ROOT}/_wav/{cls}', exist_ok=True)
    files = [p for p in sorted(glob.glob(f'{cls_dir}**/*', recursive=True))
             if os.path.isfile(p) and os.path.splitext(p)[1].lower() in AUDIO_EXT]
    for n, path in enumerate(files):
        stem = os.path.splitext(os.path.basename(path))[0]
        try:
            y, _ = librosa.load(path, sr=SR, mono=True)
        except Exception as e:
            skipped.append((path, f'cannot read: {e}')); continue
        dur = len(y) / SR
        if dur < 1.0:
            skipped.append((path, 'shorter than 1s')); continue
        if cls.startswith('a') and dur > 15:
            skipped.append((path, 'alert clip over 15s, trim it')); continue
        if cls.startswith('a') and np.sqrt(np.mean(y**2)) < 1e-3:
            skipped.append((path, 'almost silent')); continue
        if cls.startswith('a') or dur <= 12:
            pieces = [y]
        else:
            pieces = [y[s:s+8*SR] for s in range(0, len(y), 8*SR)
                      if len(y[s:s+8*SR]) >= 3*SR]
        for k, piece in enumerate(pieces):
            digest = hashlib.md5(
                np.round(piece * 32767).astype(np.int16).tobytes()
            ).hexdigest()
            if digest in seen:
                skipped.append((path, f'duplicate of {seen[digest]}')); continue
            seen[digest] = f'{cls}/{stem}'
            out = f'_wav/{cls}/{n:04d}_{stem}__{k}.wav'
            sf.write(f'{ROOT}/{out}', piece, SR, subtype='PCM_16')
            rows.append({'path': out, 'class': cls, 'group_id': group_of(stem)})

assert rows, 'No usable clips found. Check the clips/ folder and the messages above.'
manifest = pd.DataFrame(rows)
manifest.to_csv(f'{ROOT}/manifest.csv', index=False)
print(f'{len(manifest)} clips written to manifest.csv, {len(skipped)} skipped')

## Step 4b — Read the counts before training

Every class needs at least 40 **distinct sources**. If a folder shows hundreds of clips but only a handful of sources, stop: it will overfit. Set `ALLOW_LOW_SOURCES = True` only if you understand that trade-off.

In [ ]:
counts = manifest.groupby('class').agg(
    clips=('path', 'size'),
    sources=('group_id', 'nunique'),
)
print(counts.to_string())

assert manifest['class'].nunique() == len(folders), (
    f"{manifest['class'].nunique()} classes in the manifest but {len(folders)} "
    'folders on disk; a folder name changed between steps.'
)

low = counts[counts['sources'] < 40]
if len(low):
    print('\nLOW-SOURCE CLASSES:')
    print(low.to_string())
    if not ALLOW_LOW_SOURCES:
        raise SystemExit(
            'Stopped: every class needs at least 40 distinct source recordings. '
            'Add more recordings to the folders listed above, or set '
            'ALLOW_LOW_SOURCES = True and accept the overfitting risk.'
        )

if skipped:
    print(f'\nFirst 20 skipped files ({len(skipped)} total):')
    for path, reason in skipped[:20]:
        print(' ', os.path.basename(path), '->', reason)

## Step 5 — Load the manifest and build the class list

In [ ]:
manifest = pd.read_csv(f'{ROOT}/manifest.csv')
CLASSES = sorted(manifest['class'].unique())
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ALERT_CLASSES = [c for c in CLASSES if c in ALERT_WEIGHTS]
ALERT_IDS = [CLASS_TO_ID[c] for c in ALERT_CLASSES]

print(len(CLASSES), 'classes total;', len(ALERT_CLASSES), 'alert classes:', ALERT_CLASSES)
print(CLASSES)

assert len(CLASSES) == 23, (
    f'Expected exactly 23 classes (5 alert + 18 no-effect), found {len(CLASSES)}. '
    'A folder is misnamed or missing. Fix it in Drive and re-run Step 4.'
)
assert set(ALERT_CLASSES) == set(ALERT_WEIGHTS), (
    'The alert folder names do not match the names used in Step 3.'
)

## Step 6 — Load YAMNet and prove it produces real embeddings

This is the cell that catches a bad `yamnet.tflite` immediately.

In [ ]:
interp = tf.lite.Interpreter(model_path=f'{ROOT}/yamnet.tflite')
in_idx = interp.get_input_details()[0]['index']
interp.resize_tensor_input(in_idx, [WIN])
interp.allocate_tensors()
out_details = interp.get_output_details()
emb_idx = [o['index'] for o in out_details if o['shape'][-1] == EMB][0]

assert interp.get_input_details()[0]['dtype'] == np.float32, 'YAMNet input is not float32.'

def embed_window(x):
    interp.set_tensor(in_idx, x.astype(np.float32))
    interp.invoke()
    return interp.get_tensor(emb_idx).reshape(-1, EMB).mean(axis=0)

def load_clip(path):
    y, sr = sf.read(path, dtype='float32')
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y

def windows(y):
    if len(y) < WIN: y = np.pad(y, (0, WIN - len(y)))
    for s in range(0, len(y) - WIN + 1, HOP):
        yield y[s:s+WIN]

def rms(x): return float(np.sqrt(np.mean(x**2)) + 1e-9)

def mark_event_windows(ws, keep_ratio=0.3):
    """Alert clips often have a quiet tail. Keep only windows near the loudest
    part, so the model never learns 'silence = danger' from the decay of a
    loud clip."""
    levels = np.array([rms(w) for w in ws])
    return levels >= keep_ratio * levels.max()

try:
    probe = embed_window(np.zeros(WIN, dtype=np.float32))
except Exception as e:
    raise SystemExit(
        'YAMNet could not run. SUNO_dataset/yamnet.tflite is still wrong or '
        f'incompatible: {e}'
    ) from e

assert probe.shape == (EMB,), f'Unexpected embedding shape: {probe.shape}'
assert np.isfinite(probe).all(), 'YAMNet produced NaN/Inf on a silent window.'
print('YAMNet OK. Embedding shape:', probe.shape, 'L2 norm:', float(np.linalg.norm(probe)))
print('Output tensors:', [(o['index'], list(o['shape'])) for o in out_details])

## Step 7 — Split by source, 60/20/20

No source recording is allowed to appear in two splits. The asserts below prove that before training starts.

In [ ]:
y_clip = manifest['class'].map(CLASS_TO_ID).values
groups = manifest['group_id'].values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(sgkf.split(manifest, y_clip, groups))
test_idx, val_idx = folds[0][1], folds[1][1]
train_idx = np.concatenate([f[1] for f in folds[2:]])
split = np.empty(len(manifest), dtype=object)
split[train_idx], split[val_idx], split[test_idx] = 'train', 'val', 'test'
manifest['split'] = split

print(manifest.groupby(['split', 'class']).size().unstack(0).fillna(0).astype(int).to_string())
print('\nclips per split:', manifest['split'].value_counts().to_dict())

assert set(np.unique(split)) == {'train', 'val', 'test'}, 'A split is empty.'
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert (manifest.groupby('group_id')['split'].nunique() <= 1).all(), (
    'A source recording leaked across splits.'
)
for c in ALERT_CLASSES:
    present = set(manifest[manifest['class'] == c]['split'])
    assert {'train', 'test'} <= present, (
        f'{c} is missing from train or test; add more sources for this class.'
    )
print('split integrity OK')

## Step 8 — Augmentation (training split only)

In [ ]:
def aug_gain(y):
    return np.clip(y * 10**(random.uniform(-12, 6)/20), -1, 1)

def aug_noise(y, pool):
    n = random.choice(pool)
    if len(n) < len(y): n = np.tile(n, int(np.ceil(len(y)/len(n))))
    s = random.randint(0, len(n)-len(y)); n = n[s:s+len(y)]
    snr = random.uniform(5, 20)
    n = n * (rms(y) / (rms(n) * 10**(snr/20)))
    return np.clip(y + n, -1, 1)

def aug_reverb(y):
    length = int(SR * random.uniform(0.1, 0.5))
    ir = np.random.randn(length) * np.exp(-np.linspace(0, 8, length))
    ir = ir / (np.abs(ir).sum() + 1e-9)
    out = np.convolve(y, ir)[:len(y)]
    return np.clip(out * (rms(y) / rms(out)), -1, 1)

noise_classes = ['n08_traffic_horns', 'n11_home_kitchen', 'n13_speech_normal',
                 'n14_quiet_room', 'n15_phone_handling', 'n16_animals_nature']
noise_classes = [c for c in noise_classes if c in CLASSES]
assert len(noise_classes) >= 3, (
    'Fewer than 3 of the expected noise classes exist. Check the n* folder names.'
)
noise_pool = [load_clip(f'{ROOT}/{p}') for p in
              manifest[(manifest.split == 'train') & (manifest['class'].isin(noise_classes))]['path']]
assert noise_pool, 'No training noise clips found for augmentation.'

def augment(y):
    y = aug_gain(y)
    if random.random() < 0.7: y = aug_noise(y, noise_pool)
    if random.random() < 0.4: y = aug_reverb(y)
    return y

print('augmentation ready with', len(noise_pool), 'noise clips')

## Step 9 — Build and cache features

This is the slow cell (roughly 20–40 minutes for a few thousand clips). It prints progress. **Do not interrupt it** — the cache is written only when the loop finishes.

In [ ]:
CACHE = f'{ROOT}/features_v2.npz'

def build_features():
    X, Y, S, C, E = [], [], [], [], []
    started = time.time()
    total = len(manifest)
    for done, (ci, row) in enumerate(manifest.iterrows(), 1):
        if done % 100 == 0:
            print(f'{done}/{total} clips, {time.time() - started:.0f}s', flush=True)
        y = load_clip(f"{ROOT}/{row['path']}")
        cid = CLASS_TO_ID[row['class']]
        is_alert = row['class'] in ALERT_WEIGHTS
        versions = [y] + ([augment(y), augment(y)] if row['split'] == 'train' else [])
        for vi, v in enumerate(versions):
            ws = list(windows(v))
            ev = mark_event_windows(ws) if is_alert else np.ones(len(ws), dtype=bool)
            for w, flag in zip(ws, ev):
                X.append(embed_window(w)); Y.append(cid); S.append(row['split'])
                C.append(ci*10 + vi); E.append(bool(flag))
    return (np.array(X, dtype=np.float32), np.array(Y), np.array(S),
            np.array(C), np.array(E))

if os.path.exists(CACHE) and os.path.getmtime(CACHE) > os.path.getmtime(f'{ROOT}/manifest.csv'):
    d = np.load(CACHE, allow_pickle=True)
    X, Y, S, C, E = d['X'], d['Y'], d['S'], d['C'], d['E']
    print('loaded cached features', CACHE)
else:
    X, Y, S, C, E = build_features()
    np.savez(CACHE, X=X, Y=Y, S=S, C=C, E=E)
    print('wrote', CACHE)

tr = (S == 'train') & E
va_eval = (S == 'val') & E
Xtr, Ytr = X[tr], Y[tr]
Xva, Yva = X[va_eval], Y[va_eval]

assert X.shape[1] == EMB
assert np.isfinite(X).all(), 'Features contain NaN/Inf; YAMNet is still broken.'
assert len(Xtr) and len(Xva), 'Train or validation features are empty.'
print(Xtr.shape, Xva.shape)

## Step 10 — Train the head

In [ ]:
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(Xtr)
model = tf.keras.Sequential([
    tf.keras.Input((EMB,)),
    norm,
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
weights = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=Ytr)
history = model.fit(
    Xtr, Ytr, validation_data=(Xva, Yva), epochs=80, batch_size=128,
    class_weight=dict(enumerate(weights)),
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True)],
)
print('stopped at epoch', len(history.history['loss']))

## Step 11 — Per-class thresholds

Each alert class gets a threshold that keeps 99.5% of no-effect validation windows below it.

In [ ]:
pva_all_mask = (S == 'val')
pva_all = model.predict(X[pva_all_mask], verbose=0)
yva_all = Y[pva_all_mask]
neg = ~np.isin(yva_all, ALERT_IDS)
assert neg.sum() >= 20, 'Not enough no-effect validation windows to set thresholds.'

thresholds = {}
for c in ALERT_CLASSES:
    q = float(np.quantile(pva_all[neg][:, CLASS_TO_ID[c]], 0.995))
    thresholds[c] = float(min(0.95, max(0.60, q)))

print(json.dumps(thresholds, indent=2))
for c, t in thresholds.items():
    assert 0.60 <= t <= 0.95

## Step 12 — Export TFLite and verify it matches Keras

The exported file must match the Keras model to within 0.001 on the test windows. If it does not, the conversion is wrong and the file must not be used.

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
tfl = conv.convert()
tflite_path = f'{ROOT}/suno_audio_classifier_v2.tflite'
open(tflite_path, 'wb').write(tfl)

t = tf.lite.Interpreter(model_content=tfl)
t.allocate_tensors()
in_detail = t.get_input_details()[0]
out_detail = t.get_output_details()[0]
print('TFLite input :', in_detail['shape'], in_detail['dtype'])
print('TFLite output:', out_detail['shape'], out_detail['dtype'])

assert list(in_detail['shape']) == [1, EMB], (
    f"Exported input shape is {list(in_detail['shape'])}, expected [1, {EMB}]."
)
assert list(out_detail['shape']) == [1, len(CLASSES)], (
    f"Exported output shape is {list(out_detail['shape'])}, expected [1, {len(CLASSES)}]."
)
assert in_detail['dtype'] == np.float32 and out_detail['dtype'] == np.float32

ti, to = in_detail['index'], out_detail['index']
Xte, Yte = X[(S == 'test') & E], Y[(S == 'test') & E]
worst = 0.0
for k in range(min(50, len(Xte))):
    x = Xte[k:k+1]
    t.set_tensor(ti, x); t.invoke()
    worst = max(worst, float(np.abs(t.get_tensor(to) - model.predict(x, verbose=0)).max()))
print('max Keras vs TFLite difference:', worst)

assert worst < 0.001, (
    f'Keras and TFLite differ by {worst:.6f}, above the 0.001 gate. '
    'Do not ship this export.'
)

## Step 12b — Write `labels_v2.json` and package the two files

The app reads `classes` for the output-tensor width, `margin` for the ambiguity rule, and `class_meta` for alert weight / threshold / min-hits.

In [ ]:
labels_v2 = {
    'model_type': 'YAMNet_Embedding_Classifier_v2',
    'model_version': 2,
    'embedding_size': EMB,
    'sample_rate_hz': SR,
    'frame_length_sec': 0.96,
    'frame_hop_sec': 0.48,
    'classes': CLASSES,
    'margin': MARGIN,
    'class_meta': {
        c: {
            'alert_weight': ALERT_WEIGHTS.get(c, 0),
            'threshold': thresholds.get(c, 1.0),
            'min_hits_of_3': MIN_HITS.get(c, 0),
        } for c in CLASSES
    },
}
labels_path = f'{ROOT}/labels_v2.json'
with open(labels_path, 'w') as f:
    json.dump(labels_v2, f, indent=2)

release = f'{ROOT}/suno_v2_release'
os.makedirs(release, exist_ok=True)
shutil.copy2(tflite_path, release)
shutil.copy2(labels_path, release)
shutil.make_archive(release, 'zip', release)

for p in (tflite_path, labels_path):
    digest = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    print(os.path.basename(p), os.path.getsize(p), 'bytes')
    print('  sha256:', digest)
print('\nDownload folder:', release)
print('Download zip   :', release + '.zip')

## Step 13 — Replay the app's actual decision rule on held-out test clips

This is the gate that decides whether to ship. It replays the threshold + margin + repeat rule on clips the model has never seen.

In [ ]:
def alert_label(p, thresholds):
    order = np.argsort(p)[::-1]
    top, second = order[0], order[1]
    name = CLASSES[top]
    if name not in thresholds: return None
    if p[top] < thresholds[name]: return None
    if p[top] - p[second] < MARGIN: return None
    return name

def clip_fires(P):
    labels = [alert_label(p, thresholds) for p in P]
    fired = set()
    for i in range(len(labels)):
        span = labels[max(0, i-2):i+1]
        for c in set(x for x in span if x):
            if span.count(c) >= MIN_HITS[c]:
                fired.add(c)
    return fired

te_mask = (S == 'test') & (C % 10 == 0)
Pte = model.predict(X[te_mask], verbose=0)
df = pd.DataFrame({'clip': C[te_mask], 'cls': [CLASSES[i] for i in Y[te_mask]]})
df['row'] = np.arange(len(df))
assert len(df), 'No held-out test windows; check the split.'

results = []
for clip, g in df.groupby('clip', sort=False):
    fired = clip_fires(Pte[g['row'].values])
    results.append({'clip': clip, 'cls': g['cls'].iloc[0],
                    'n_windows': len(g), 'fired': fired})
res = pd.DataFrame(results)

recall = {}
recall_counts = {}
for c in ALERT_CLASSES:
    subset = res[res.cls == c]
    recall_counts[c] = len(subset)
    recall[c] = float(np.mean([c in f for f in subset.fired])) if len(subset) else 0.0

neg_res = res[~res.cls.isin(ALERT_CLASSES)]
false_clip_rate = float(np.mean([len(f) > 0 for f in neg_res.fired])) if len(neg_res) else 1.0

print('per-class recall:', {k: round(v, 3) for k, v in recall.items()})
print('held-out clips per alert class:', recall_counts)
print('false alert rate on no-effect test clips: %.2f%%' % (100 * false_clip_rate))
print()
gates = {}
for c in ALERT_CLASSES:
    ok = recall.get(c, 0) >= 0.85
    gates[c] = 'PASS' if ok else 'FAIL'
    print(c, '->', gates[c], f'(recall {recall.get(c, 0):.3f}, {recall_counts[c]} clips)')
false_ok = false_clip_rate < 0.01
print('overall false-alert gate ->', 'PASS' if false_ok else 'FAIL')

summary = {
    'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'classes': len(CLASSES),
    'alert_classes': ALERT_CLASSES,
    'clip_counts': {k: int(v) for k, v in manifest['class'].value_counts().items()},
    'split_counts': {k: int(v) for k, v in manifest['split'].value_counts().items()},
    'thresholds': thresholds,
    'margin': MARGIN,
    'recall': recall,
    'recall_clip_counts': {k: int(v) for k, v in recall_counts.items()},
    'false_alert_rate': false_clip_rate,
    'keras_vs_tflite_max_diff': worst,
    'gates': gates,
    'false_alert_gate': 'PASS' if false_ok else 'FAIL',
}
summary_path = f'{ROOT}/training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSaved', summary_path)
print('\nSTOP HERE if any gate says FAIL. Send the summary back; do not install v2.')

## What to send back to Qoder

Copy the printed output from these cells:

1. **Step 4b** — the class/source count table.
2. **Step 11** — the threshold dictionary.
3. **Step 12** — the TFLite input/output shapes and `max Keras vs TFLite difference`.
4. **Step 13** — per-class recall, false-alert rate, and the gate lines.
5. **Step 12b** — the two SHA-256 lines (so the exact files can be verified).

Then put these two files in the repo (not done automatically):

- `SUNO_dataset/suno_audio_classifier_v2.tflite` → `assets/ml/suno_audio_classifier_v2.tflite`
- `SUNO_dataset/labels_v2.json` → `assets/ml/labels_v2.json`

Only after **every** gate is PASS and the live tests in Part B pass should `useModelV2` be flipped to `true`.